In [1]:
# Loading  libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import levene, ttest_ind, mannwhitneyu


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier




import mlflow



from pathlib import Path

In [2]:
file_path = Path("../data/Womens Clothing E-Commerce Reviews.csv")

df = pd.read_csv(file_path)
df_orignal = df.copy()
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [3]:
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')


Number of rows: 23486
Number of columns: 11


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Unnamed: 0               23486 non-null  int64
 1   Clothing ID              23486 non-null  int64
 2   Age                      23486 non-null  int64
 3   Title                    19676 non-null  str  
 4   Review Text              22641 non-null  str  
 5   Rating                   23486 non-null  int64
 6   Recommended IND          23486 non-null  int64
 7   Positive Feedback Count  23486 non-null  int64
 8   Division Name            23472 non-null  str  
 9   Department Name          23472 non-null  str  
 10  Class Name               23472 non-null  str  
dtypes: int64(6), str(5)
memory usage: 9.5 MB


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,23486.0,11742.500000,6779.968547,0.0,5871.25,11742.5,17613.75,23485.0
Clothing ID,23486.0,918.118709,203.298980,0.0,861.00,936.0,1078.00,1205.0
Age,23486.0,43.198544,12.279544,18.0,34.00,41.0,52.00,99.0
Rating,23486.0,4.196032,1.110031,1.0,4.00,5.0,5.00,5.0
Recommended IND,23486.0,0.822362,0.382216,0.0,1.00,1.0,1.00,1.0
Positive Feedback Count,23486.0,2.535936,5.702202,0.0,0.00,1.0,3.00,122.0


**Notes**:  
* Missing values in a `Title`, `Review Text`, `Division Name`, `Department Name`, `Class Name`, will need further inspection
* the max value in 	`Positive Feedback Count` might be an entry error, will need to be inspected
* No negative values in the data
* The mean values of `Rating` and `Positive Feedback Count` are larger than the median values indicating a right skew distribution.

In [6]:
def data_summary(df: pd.DataFrame) -> pd.DataFrame:
    missing_perc = round(df.isna().mean() * 100, 2)
    unique_values = df.nunique()
    col_types = df.dtypes
    duplicates = df.duplicated().sum()

    print(f'Number of duplicates entires : {duplicates}')
    summary_df = pd.DataFrame({
        'missing_perc' : missing_perc,
        'unique_values' : unique_values,
        'col_type' : col_types
    })
    return summary_df[summary_df['missing_perc'] > 0.0].sort_values(by='missing_perc', ascending=False)

data_summary(df=df)

Number of duplicates entires : 0


,missing_perc,unique_values,col_type
Title,16.22,13993,str
Review Text,3.60,22634,str
Division Name,0.06,3,str
Department Name,0.06,6,str
Class Name,0.06,20,str


**Notes**:  
* No duplicated entries
* `Title` column has the largest number of missing values
* `Positive Feedback Count` is right skewed. Most items get 0–7 upvotes, so anything with 8+ upvotes gets flagged as an extreme value.
* `Age` Only extreme ages outside 7 to 79 years old are flagged.